# Building a RAG Pipeline: Step-by-Step

**Pipeline Flow:** Documents → Chunks → Embeddings → Vector DB → Query → Retrieve → Generate Answer

First, we need to import our tools and set up our API keys. 
* **LangChain:** The framework that ties everything together.
* **FAISS:** Our vector database for fast similarity searches.
* **HuggingFace & OpenAI:** The AI models we'll use for understanding and generating text.

In [ ]:
import os
import uuid
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

# Replace with your actual key
my_api_key = "YOUR_API_KEY_HERE"
os.environ["OPENAI_API_KEY"] = my_api_key

## Step 1: Define the Knowledge Base
Large Language Models (LLMs) only know what they were trained on. To get them to answer questions about our specific data, we have to provide it. 

In a real application, you would load these from PDFs, Notion, or databases. Here, we are using a simple list of dictionaries where each document has a unique ID, a title, and the actual text.

In [ ]:
RAW_DOCS = [
    {
        "id": "hr-policy",
        "title": "HR Leave Policy 2024",
        "text": "\nOur company provides 20 days of paid annual leave per year for full-time employees.\nEmployees can carry over up to 5 unused leave days to the next year.\nSick leave is separate and covered under the Health & Wellness policy.\nFor special circumstances, additional unpaid leave may be approved by HR.\n"
    },
    {
        "id": "rag-notes",
        "title": "RAG System Overview",
        "text": "\nRetrieval-Augmented Generation (RAG) combines a retriever and a generator.\nThe retriever finds relevant document chunks using embeddings and vector search.\nThe generator, usually a large language model, uses those chunks to answer questions.\nRAG reduces hallucinations by grounding answers in real documents.\n"
    },
    {
        "id": "product-analytics",
        "title": "Product Analytics Basics",
        "text": "\nProduct analytics focuses on how users interact with a product.\nKey metrics include activation, engagement, retention, and conversion.\nTeams use these insights to improve onboarding, features, and user experience.\nAccurate tracking and event naming are critical for trustworthy analytics.\n"
    }
]

## Step 2: Document Chunking
**Why chunk?** Entire rulebooks or databases won't fit into an LLM's context window. Chunking breaks documents down into bite-sized, searchable pieces while preserving their core meaning.

We use a `chunk_size` of 300 characters, with a 60-character `chunk_overlap`. The overlap ensures we don't accidentally split a sentence or thought right down the middle, which would destroy its context.

In [ ]:
def build_chunks(raw_docs):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=60,
        separators=["\n\n", "\n", ". ", " "]
    )

    chunks = []
    for doc in raw_docs:
        for chunk_text in splitter.split_text(doc["text"]):
            if chunk_text.strip():
                # We attach metadata so we always know where a chunk came from!
                chunks.append(
                    {
                        "page_content": chunk_text.strip(),
                        "metadata": {
                            "source_id": doc["id"],
                            "title": doc["title"]
                        }
                    }
                )
    return chunks

# Execute the chunking
chunks = build_chunks(RAW_DOCS)
print(f"Created {len(chunks)} chunks.")
# Run this cell, then create a new cell and type `chunks[0]` to see what a chunk looks like!

## Step 3: Embeddings & Vector Storage
Computers don't understand English—they understand math. **Embeddings** convert our text chunks into arrays of numbers (vectors). If two pieces of text have similar meanings (e.g., "vacation days" and "annual leave"), their resulting vectors will be mathematically close to each other.

We are using HuggingFace's `all-MiniLM-L6-v2` to create the vectors, and Facebook's **FAISS** database to store them so we can search them instantly.

In [ ]:
def build_vectorstore(chunks):
    # Initialize the embedding model
    embed_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    # Separate the text and metadata
    texts = [c["page_content"] for c in chunks]
    metadatas = [c["metadata"] for c in chunks]

    # Create the FAISS vector database
    vectordb = FAISS.from_texts(
        texts=texts,
        embedding=embed_model,
        metadatas=metadatas
    )
    return vectordb

# Build the database
vectordb = build_vectorstore(chunks)
print("Vector database built successfully!")

## Step 4: System Assembly
Now we wire the pieces together. 
1. **The Retriever:** We configure our vector database to return the 3 most relevant chunks (`k=3`) for any given question.
2. **The LLM:** We set up OpenAI's `gpt-4o-mini`. We set `temperature=0` because we want factual, deterministic answers, not creative storytelling.
3. **The Prompt:** We give the LLM strict instructions to *only* use the provided context to answer the question.

In [ ]:
def build_qa_system(vectordb):
    # Set up the retriever to find the top 3 chunks
    retriever = vectordb.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 3}
    )

    system = {"retriever": retriever}
    
    try:
        llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=0
        )
        
        prompt_template = PromptTemplate.from_template(
            """Based on the following context, answer the question.
            
Context:
{context}

Question: {question}

Answer:"""
        )
        system.update({"llm": llm, "prompt": prompt_template})
        print("OpenAI LLM successfully configured!")
    except Exception as e:
        print(f"OpenAI setup failed: {e}")
        
    return system

# Assemble the system
qa_system = build_qa_system(vectordb)

## Step 5: Ask Questions!
When we ask a question, the pipeline:
1. Converts our question into an embedding.
2. Searches the vector database for the closest matching chunks.
3. Shoves those chunks into the Prompt Template.
4. Sends the whole prompt to OpenAI to generate a natural, readable answer.

In [ ]:
def run_query(qa_system, question: str):
    trace_id = str(uuid.uuid4())
    print(f"\n=== TRACE ID: {trace_id} ===")
    print(f"User Question: {question}\n")

    # 1. Retrieve relevant chunks
    source_docs = qa_system["retriever"].invoke(question)

    print("--- Retrieved Context Chunks ---")
    for i, doc in enumerate(source_docs, start=1):
        meta = doc.metadata
        preview = doc.page_content[:160].replace("\n", " ")
        print(f"[{i}] {meta.get('title')} (source_id={meta.get('source_id')})")
        print(f"    {preview}...\n")

    # 2. Generate the answer
    if "llm" in qa_system and "prompt" in qa_system:
        context = "\n\n".join([doc.page_content for doc in source_docs])
        formatted_prompt = qa_system["prompt"].format(context=context, question=question)
        
        answer = qa_system["llm"].invoke(formatted_prompt).content

        print("--- Final Answer ---")
        print(answer)

In [ ]:
# Try changing these questions to see how the retriever pulls different chunks!
run_query(qa_system, "How many days of annual leave do we get?")
run_query(qa_system, "What is RAG and why is it useful?")